In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Install BigQuery library
!pip install google-cloud-bigquery -q

import pandas as pd
from google.cloud import bigquery
import os

# Set path to your service account key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/social-ads-project/social-ads-analytics-85610564b570.json"

# Initialize BigQuery client
client = bigquery.Client(project="social-ads-analytics")

# Define your dataset
dataset_id = "social_ads_raw"
base_path = "/content/drive/MyDrive/social-ads-project/"

# Load each CSV
tables = {
    "users": "users.csv",
    "campaigns": "campaigns.csv",
    "ads": "ads.csv",
    "ad_events": "ad_events.csv"
}

for table_name, file_name in tables.items():
    df = pd.read_csv(base_path + file_name)
    table_ref = f"social-ads-analytics.{dataset_id}.{table_name}"
    job = client.load_table_from_dataframe(df, table_ref)
    job.result()
    print(f"✓ Loaded {len(df)} rows into {table_name}")

✓ Loaded 10000 rows into users
✓ Loaded 50 rows into campaigns
✓ Loaded 200 rows into ads
✓ Loaded 400000 rows into ad_events


In [3]:
# Verify all tables loaded correctly
for table_name in tables.keys():
    table = client.get_table(f"social-ads-analytics.social_ads_raw.{table_name}")
    print(f"✓ {table_name}: {table.num_rows} rows, {len(table.schema)} columns")

✓ users: 10000 rows, 7 columns
✓ campaigns: 50 rows, 6 columns
✓ ads: 200 rows, 7 columns
✓ ad_events: 400000 rows, 7 columns


In [4]:
query = "SELECT column_name FROM social-ads-analytics.social_ads_raw.INFORMATION_SCHEMA.COLUMNS WHERE table_name = 'campaigns'"
df = client.query(query).to_dataframe()
print(df)

     column_name
0    campaign_id
1           name
2     start_date
3       end_date
4  duration_days
5   total_budget


In [5]:
for table in ['ads', 'ad_events']:
    query = f"SELECT column_name FROM `social-ads-analytics.social_ads_raw.INFORMATION_SCHEMA.COLUMNS` WHERE table_name = '{table}'"
    df = client.query(query).to_dataframe()
    print(f"\n{table} columns:")
    print(df)


ads columns:
        column_name
0             ad_id
1       campaign_id
2       ad_platform
3           ad_type
4     target_gender
5  target_age_group
6  target_interests

ad_events columns:
   column_name
0     event_id
1        ad_id
2      user_id
3    timestamp
4  day_of_week
5  time_of_day
6   event_type


In [6]:
query = """
SELECT
    min(start_date) as earliest_start,
    max(end_date) as latest_end
FROM `social-ads-analytics.social_ads_raw.campaigns`
"""
df = client.query(query).to_dataframe()
print(df)

  earliest_start  latest_end
0     2025-02-13  2025-10-12


In [7]:
query = "SELECT column_name FROM `social-ads-analytics.social_ads_raw.INFORMATION_SCHEMA.COLUMNS` WHERE table_name = 'users'"
df = client.query(query).to_dataframe()
print(df)

   column_name
0      user_id
1  user_gender
2     user_age
3    age_group
4      country
5     location
6    interests


In [8]:
query = """
SELECT
    campaign_phase,
    count(*) as total_events,
    count(distinct user_id) as unique_users,
    count(distinct campaign_id) as campaigns
FROM `social-ads-analytics.social_ads_dbt.fct_campaign_performance`
GROUP BY campaign_phase
ORDER BY total_events desc
"""
df = client.query(query).to_dataframe()
print(df)

  campaign_phase  total_events  unique_users  campaigns
0  Full campaign        237615          9950         48
1    4 months in        166352          9950         48


In [9]:
query = """
SELECT * FROM `social-ads-analytics.social_ads_dbt.fct_campaign_performance`
"""
df = client.query(query).to_dataframe()
df.to_csv('/content/drive/MyDrive/social-ads-project/fct_campaign_performance.csv', index=False)
print(f"Exported {len(df)} rows")

Exported 403967 rows


In [10]:
query = """
SELECT DISTINCT location, COUNT(*) as count
FROM `social-ads-analytics.social_ads_raw.users`
GROUP BY location
ORDER BY count DESC
LIMIT 20
"""
df = client.query(query).to_dataframe()
print(df)

             location  count
0        West Michael     15
1       North Michael     13
2        Port Michael     11
3          North John     11
4         New Michael     11
5          Lake David     11
6           New David     10
7           Port John     10
8         North David      8
9        East Michael      8
10       Michaelhaven      8
11        East Robert      8
12      South William      8
13        South James      8
14  North Christopher      8
15       New Jennifer      7
16       Jenniferland      7
17        New Jessica      7
18     East Stephanie      7
19         Lake James      7
